# 23 · Working with JSON

Modern apps store semi-structured JSON in a column. SQLite's JSON functions let
you query and build it with SQL.
- extract values: `json_extract`, the `->` and `->>` operators
- build JSON: `json_object`, `json_array`, `json_group_array`
- expand JSON into rows: `json_each` (a table-valued function)
- modify JSON: `json_set`
- indexing JSON with an expression index

> Requires SQLite 3.38+ for the `->>` operator (your Python 3.12 build is newer).
> Uses a throwaway `demo_events` table.

In [ ]:
# ▶ Run this cell first. It loads JupySQL and connects to the SQLite database.
%load_ext sql
from sqlalchemy import create_engine
import os

# Works whether the notebook's working dir is the repo root or notebooks/
db_path = 'data/retail.db' if os.path.exists('data/retail.db') else '../data/retail.db'
engine = create_engine(f'sqlite:///{db_path}')

%config SqlMagic.autopandas = True      # results come back as pandas DataFrames
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100

%sql engine
print('Connected to', db_path)

## Store some JSON

In [ ]:
%%sql
DROP TABLE IF EXISTS demo_events;
CREATE TABLE demo_events (
    id      INTEGER PRIMARY KEY,
    payload TEXT   -- JSON stored as text
);
INSERT INTO demo_events (payload) VALUES
    ('{"user":"alice","action":"login","amount":0,"tags":["web","mobile"]}'),
    ('{"user":"bob","action":"purchase","amount":42.5,"tags":["web"]}'),
    ('{"user":"carla","action":"purchase","amount":19.0,"tags":["mobile","promo"]}');
SELECT * FROM demo_events;

## Extracting values
- `json_extract(payload, '$.user')` — path-based extraction
- `payload -> '$.user'` — returns JSON
- `payload ->> '$.user'` — returns a plain SQL text/number value (usually what you want)

In [ ]:
%%sql
SELECT id,
       json_extract(payload, '$.user')   AS user,
       payload ->> 'action'              AS action,
       CAST(payload ->> 'amount' AS REAL) AS amount
FROM demo_events;

## Filter and aggregate on JSON fields
Total purchase amount per action:

In [ ]:
%%sql
SELECT payload ->> 'action' AS action,
       ROUND(SUM(payload ->> 'amount'), 2) AS total_amount,
       COUNT(*) AS events
FROM demo_events
GROUP BY action;

## Expand a JSON array into rows with `json_each`
`json_each` is a table-valued function — join it to a table to unnest arrays.
Here we explode each event's `tags` array into one row per tag, then count tags:

In [ ]:
%%sql
SELECT j.value AS tag, COUNT(*) AS uses
FROM demo_events e, json_each(e.payload, '$.tags') AS j
GROUP BY j.value
ORDER BY uses DESC, tag;

## Build JSON from rows
`json_object` builds an object; `json_group_array` aggregates rows into a JSON array:

In [ ]:
%%sql
SELECT json_object(
           'category', c.category_name,
           'products', json_group_array(p.product_name)
       ) AS category_json
FROM categories c
JOIN products p ON p.category_id = c.category_id
WHERE c.category_name = 'Books'
GROUP BY c.category_name;

## Modify JSON with `json_set`
Add/replace a field, returning new JSON:

In [ ]:
%%sql
SELECT id,
       json_set(payload, '$.processed', json('true')) AS updated
FROM demo_events
WHERE id = 1;

## Index a JSON field (expression index)
Speeds up filters on a specific JSON path by indexing the extracted value:

In [ ]:
%%sql
DROP INDEX IF EXISTS demo_idx_event_user;
CREATE INDEX demo_idx_event_user ON demo_events(json_extract(payload, '$.user'));
EXPLAIN QUERY PLAN
SELECT * FROM demo_events WHERE json_extract(payload, '$.user') = 'bob';

## Practice

**✏️ Exercise 1.** From `demo_events` (still populated above), list only the events whose action is 'purchase', showing user and amount from the JSON.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
SELECT payload ->> 'user' AS user, payload ->> 'amount' AS amount
FROM demo_events
WHERE payload ->> 'action' = 'purchase';

## Clean up

In [ ]:
%%sql
DROP INDEX IF EXISTS demo_idx_event_user;
DROP TABLE IF EXISTS demo_events;
SELECT 'cleaned up' AS status;

### ✅ Recap
`json_extract`/`->>` read fields, `json_each` unnests arrays into rows,
`json_object`/`json_group_array` build JSON, `json_set` edits it, and expression
indexes speed up JSON-path filters.

**Next:** `24_triggers_and_advanced_views.ipynb`.